In [2]:
import pandas as pd

In [3]:
df = pd.read_csv(r"D:\University\Third Year\Semester Two\Natural Language Processing\Project\spam dataset.csv", encoding='latin-1')
print(df.shape)

(5572, 5)


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   v1          5572 non-null   object
 1   v2          5572 non-null   object
 2   Unnamed: 2  50 non-null     object
 3   Unnamed: 3  12 non-null     object
 4   Unnamed: 4  6 non-null      object
dtypes: object(5)
memory usage: 217.8+ KB


In [5]:
df.isnull().sum()

v1               0
v2               0
Unnamed: 2    5522
Unnamed: 3    5560
Unnamed: 4    5566
dtype: int64

In [6]:
print(df.head())
print(df.columns)

     v1                                                 v2 Unnamed: 2  \
0   ham  Go until jurong point, crazy.. Available only ...        NaN   
1   ham                      Ok lar... Joking wif u oni...        NaN   
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...        NaN   
3   ham  U dun say so early hor... U c already then say...        NaN   
4   ham  Nah I don't think he goes to usf, he lives aro...        NaN   

  Unnamed: 3 Unnamed: 4  
0        NaN        NaN  
1        NaN        NaN  
2        NaN        NaN  
3        NaN        NaN  
4        NaN        NaN  
Index(['v1', 'v2', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4'], dtype='object')


In [7]:
df.describe()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
count,5572,5572,50,12,6
unique,2,5169,43,10,5
top,ham,"Sorry, I'll call later","bt not his girlfrnd... G o o d n i g h t . . .@""","MK17 92H. 450Ppw 16""","GNT:-)"""
freq,4825,30,3,2,2


In [8]:
#drop nulls
df = df.dropna(subset=['v1', 'v2'])

In [9]:
#drop duplicats
df = df.drop_duplicates()

In [10]:
#reset index
df = df.reset_index(drop=True)

In [11]:
df = df[['v1', 'v2']]

In [12]:
df.columns = ['label', 'message']

In [13]:
df['label'] = df['label'].map({'ham': 0, 'spam': 1})

In [14]:
df['message_length'] = df['message'].apply(len)

**Data Preprocessing**

Data cleaning was performed by removing missing values and duplicate records.
Additionally , a new feature representing message length was added to enhance analysis.

In [15]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split( df['message'], df['label'], test_size=0.2, random_state=42 )

In [16]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(stop_words='english')

X_train = vectorizer.fit_transform(X_train)
X_test = vectorizer.transform(X_test)

**Naive Bayes**

In [17]:
from sklearn.naive_bayes import MultinomialNB

model_nb= MultinomialNB()
model_nb.fit(X_train, y_train)
print("Accuracy:", model_nb.score(X_test, y_test))

Accuracy: 0.9661508704061895


In [18]:
msg = ["Win a free prize now"]

msg_vec = vectorizer.transform(msg)

print(model_nb.predict(msg_vec))

[1]


**Logistic Regression**

In [19]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression()
lr.fit(X_train, y_train)

print("Logistic Regression Accuracy:", lr.score(X_test, y_test))

Logistic Regression Accuracy: 0.9642166344294004


**SVM**

In [20]:
from sklearn.svm import LinearSVC

svm = LinearSVC()
svm.fit(X_train, y_train)

print("SVM Accuracy:", svm.score(X_test, y_test))

SVM Accuracy: 0.9825918762088974


**Decision Tree**

In [21]:
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier()
dt.fit(X_train, y_train)

print("Decision Tree Accuracy:", dt.score(X_test, y_test))

Decision Tree Accuracy: 0.9642166344294004


**Random Forest**

In [22]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier()
rf.fit(X_train, y_train)

print("Random Forest Accuracy:", rf.score(X_test, y_test))

Random Forest Accuracy: 0.9729206963249516


**XGBOOST**

In [23]:
!pip install xgboost

'pip' is not recognized as an internal or external command,
operable program or batch file.


In [24]:
from xgboost import XGBClassifier

xgb = XGBClassifier(eval_metric='logloss')

xgb.fit(X_train, y_train)

print("XGBoost Accuracy:", xgb.score(X_test, y_test))

XGBoost Accuracy: 0.9738878143133463


In [25]:
models = {
    "Naive Bayes": model_nb,
    "Logistic Regression": lr,
    "SVM": svm,
    "Decision Tree": dt,
    "Random Forest": rf,
    "XGBoost": xgb
}

results = []

for name, m in models.items():
    acc = m.score(X_test, y_test)
    results.append([name, acc])

df_results = pd.DataFrame(results, columns=["Model", "Accuracy"])

print(df_results)

                 Model  Accuracy
0          Naive Bayes  0.966151
1  Logistic Regression  0.964217
2                  SVM  0.982592
3        Decision Tree  0.964217
4        Random Forest  0.972921
5              XGBoost  0.973888


**Results & Discussion**

-The experimental results show that SVM and Random Forest achieved the highest accuracy among all tested models, while Logistic Regression showed slightly lower performance. However, all models performed consistently well, with accuracy ranging between 0.95 and 0.97

**Conclusion**

-The system successfully classifies spam messages using multiple machine learning models with high accuracy

-All models were trained using the same preprocessing pipeline (TF-IDF vectorization)

**GUI**

In [26]:
import tkinter as tk

def convert(x):
    if x == 1:
        return "Spam 🚨", "red"
    else:
        return "Ham ✅", "green"

def predict():
    text = entry.get()

    if text.strip() == "":
        result.config(text="Please enter a message ❗", fg="black")
        return

    vec = vectorizer.transform([text])

    nb_text, nb_color = convert(model_nb.predict(vec)[0])
    lr_text, lr_color = convert(lr.predict(vec)[0])
    svm_text, svm_color = convert(svm.predict(vec)[0])
    dt_text, dt_color = convert(dt.predict(vec)[0])
    rf_text, rf_color = convert(rf.predict(vec)[0])
    xgb_text, xgb_color = convert(xgb.predict(vec)[0])

    nb_res.config(text="Naive Bayes: " + nb_text, fg=nb_color)
    lr_res.config(text="Logistic Regression: " + lr_text, fg=lr_color)
    svm_res.config(text="SVM: " + svm_text, fg=svm_color)
    dt_res.config(text="Decision Tree: " + dt_text, fg=dt_color)
    rf_res.config(text="Random Forest: " + rf_text, fg=rf_color)
    xgb_res.config(text="XGBoost: " + xgb_text, fg=xgb_color)

# ===== GUI =====
root = tk.Tk()
root.title("Spam Detection System")
root.geometry("420x380")

entry = tk.Entry(root, width=45)
entry.pack(pady=10)

tk.Button(root, text="Process", command=predict).pack()

result = tk.Label(root, text="")
result.pack(pady=10)

nb_res = tk.Label(root, text="Naive Bayes:")
nb_res.pack()

lr_res = tk.Label(root, text="Logistic Regression:")
lr_res.pack()

svm_res = tk.Label(root, text="SVM:")
svm_res.pack()

dt_res = tk.Label(root, text="Decision Tree:")
dt_res.pack()

rf_res = tk.Label(root, text="Random Forest:")
rf_res.pack()

xgb_res = tk.Label(root, text="XGBoost:")
xgb_res.pack()

root.mainloop()